In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (StructType, StructField, StringType, DecimalType, IntegerType, DateType)
from pyspark.sql import functions as F
from pyspark.sql.functions import round

spark = SparkSession.builder.appName("beverage-sales").getOrCreate()

schema = StructType([
    StructField("Order_ID", StringType(), True),
    StructField("Customer_ID", StringType(), True),
    StructField("Customer_Type", StringType(), True),
    StructField("Product", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Unit_Price", DecimalType(10, 2), True),
    StructField("Quantity", IntegerType(), True),
    StructField("Discount", DecimalType(5, 2), True),
    StructField("Total_Price", DecimalType(12, 2), True),
    StructField("Region", StringType(), True),
    StructField("Order_Date", DateType(), True)
])

df_raw = spark.read \
    .format("csv") \
    .option("header", True) \
    .option("delimiter", ",") \
    .option("mode", "FAILFAST") \
    .option("dateFormat", "yyyy-MM-dd") \
    .schema(schema) \
    .load("/home/rgerc/Projects/beverage-sales/data/raw/synthetic_beverage_sales_data.csv")

#------------------------Data Cleaning------------------------#

print("Verify if there are any null values in the dataset:")
df_raw.select([F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_raw.columns
]).show()

print("Check for duplicate Order_IDs (it could make sense to have multiple entries for the same order):")
df_raw.groupBy("Order_ID").count().orderBy(F.desc("count")).show()

print("Check if Total_Price is correctly calculated:")
df_check = df_raw.withColumn(
    "Expected_Total",
    F.round(
        F.col("Unit_Price") * F.col("Quantity") * (1 - F.col("Discount")),
        2
    )
)

print("Show rows where Total_Price does not match the expected calculation:")
df_check.filter(
    F.abs(F.col("Expected_Total") - F.col("Total_Price")) > 0.01
).show()

print("Check for negative or zero values in Quantity and Unit_Price, and invalid Discount values:")
df_raw.filter(F.col("Quantity") <= 0).show()

df_raw.filter(F.col("Unit_Price") <= 0).show()

df_raw.filter((F.col("Discount") < 0) | (F.col("Discount") > 1)).show()

print("Some columns need to be derived for easier analysis:")
df_clean = df_raw \
    .withColumn("year", F.year("Order_Date")) \
    .withColumn("month", F.month("Order_Date")) \
    .withColumn("gross_sales", F.col("Unit_Price") * F.col("Quantity")) \
    .withColumn("discount_value", F.col("gross_sales") * F.col("Discount")) \
    .withColumn("net_sales", F.col("Total_Price"))

df_clean.show(5)


Verify if there are any null values in the dataset:


+--------+-----------+-------------+-------+--------+----------+--------+--------+-----------+------+----------+
|Order_ID|Customer_ID|Customer_Type|Product|Category|Unit_Price|Quantity|Discount|Total_Price|Region|Order_Date|
+--------+-----------+-------------+-------+--------+----------+--------+--------+-----------+------+----------+
|       0|          0|            0|      0|       0|         0|       0|       0|          0|     0|         0|
+--------+-----------+-------------+-------+--------+----------+--------+--------+-----------+------+----------+

Check for duplicate Order_IDs (it could make sense to have multiple entries for the same order):


+--------+-----+
|Order_ID|count|
+--------+-----+
|ORD18686|    5|
|ORD13074|    5|
|ORD19562|    5|
|ORD34749|    5|
|ORD21348|    5|
|ORD14199|    5|
|ORD30773|    5|
| ORD5068|    5|
|ORD14768|    5|
|ORD15600|    5|
|ORD20013|    5|
| ORD5252|    5|
|ORD43030|    5|
|ORD15991|    5|
| ORD1822|    5|
| ORD5365|    5|
|ORD19920|    5|
|ORD16403|    5|
| ORD2815|    5|
| ORD6158|    5|
+--------+-----+
only showing top 20 rows
Check if Total_Price is correctly calculated:
Show rows where Total_Price does not match the expected calculation:


+--------+-----------+-------------+-------+--------+----------+--------+--------+-----------+------+----------+--------------+
|Order_ID|Customer_ID|Customer_Type|Product|Category|Unit_Price|Quantity|Discount|Total_Price|Region|Order_Date|Expected_Total|
+--------+-----------+-------------+-------+--------+----------+--------+--------+-----------+------+----------+--------------+
+--------+-----------+-------------+-------+--------+----------+--------+--------+-----------+------+----------+--------------+

Check for negative or zero values in Quantity and Unit_Price, and invalid Discount values:


+--------+-----------+-------------+-------+--------+----------+--------+--------+-----------+------+----------+
|Order_ID|Customer_ID|Customer_Type|Product|Category|Unit_Price|Quantity|Discount|Total_Price|Region|Order_Date|
+--------+-----------+-------------+-------+--------+----------+--------+--------+-----------+------+----------+
+--------+-----------+-------------+-------+--------+----------+--------+--------+-----------+------+----------+



+--------+-----------+-------------+-------+--------+----------+--------+--------+-----------+------+----------+
|Order_ID|Customer_ID|Customer_Type|Product|Category|Unit_Price|Quantity|Discount|Total_Price|Region|Order_Date|
+--------+-----------+-------------+-------+--------+----------+--------+--------+-----------+------+----------+
+--------+-----------+-------------+-------+--------+----------+--------+--------+-----------+------+----------+



+--------+-----------+-------------+-------+--------+----------+--------+--------+-----------+------+----------+
|Order_ID|Customer_ID|Customer_Type|Product|Category|Unit_Price|Quantity|Discount|Total_Price|Region|Order_Date|
+--------+-----------+-------------+-------+--------+----------+--------+--------+-----------+------+----------+
+--------+-----------+-------------+-------+--------+----------+--------+--------+-----------+------+----------+

Some columns need to be derived for easier analysis:
+--------+-----------+-------------+------------------+-----------+----------+--------+--------+-----------+-----------------+----------+----+-----+-----------+--------------+---------+
|Order_ID|Customer_ID|Customer_Type|           Product|   Category|Unit_Price|Quantity|Discount|Total_Price|           Region|Order_Date|year|month|gross_sales|discount_value|net_sales|
+--------+-----------+-------------+------------------+-----------+----------+--------+--------+-----------+--------------